In [ ]:
import pandas as pd
import os
from src.utils import images_to_video

In [ ]:
def analyze_results(detected_frames_csv, ground_truth_csv, excluded_frames_csv, image_folder, output_folder):    
    """
    Analyze detection results against ground truth data
    """
    output_folder = output_folder + "false_positives/"
    os.makedirs(output_folder, exist_ok=True)

    # Load detected frames and ground truth data
    detected_frames = pd.read_csv(detected_frames_csv)
    ground_truth = pd.read_csv(ground_truth_csv)
    exclusion_ranges = pd.read_csv(excluded_frames_csv)

    TOLERENCE_THRESHOLD = 10
    
    # Rename columns
    detected_frames.columns = ['overtaking_id', 'track_id', 'first_frame', 'end_frame', 'vehicle_class']
    ground_truth.columns = ['overtaking_frame', 'lane']
    ground_truth_frames = ground_truth['overtaking_frame'].tolist()

    # Filter out detections that fall within exclusion ranges
    valid_detections = []
    for _, detection in detected_frames.iterrows():
        is_excluded = False
        for _, exclusion in exclusion_ranges.iterrows():
            if (detection['first_frame'] >= exclusion['start_frame'] and 
                detection['end_frame'] <= exclusion['end_frame']):
                is_excluded = True
                break
        if not is_excluded:
            valid_detections.append(detection)
    
    detected_frames = pd.DataFrame(valid_detections)

    # Remove all detected frame ranges that are less than 5 frames
    if not detected_frames.empty:
        detected_frames = detected_frames[
            (detected_frames['end_frame'] - detected_frames['first_frame']) >= 5
        ].reset_index(drop=True)

    # Initialize tracking variables
    matched_detections = {}  # track_id: matched_gt_frame
    matched_ground_truth = {}  # gt_frame: matched_track_id
    false_positives = []
    false_negatives = []

    # Match ground truth frames with detections
    for gt_frame in ground_truth_frames:
        match_found = False
        if not detected_frames.empty:
            for idx, detection in detected_frames.iterrows():
                track_id = detection['track_id']
                if (detection['first_frame'] - TOLERENCE_THRESHOLD <= gt_frame <= 
                    detection['end_frame'] + TOLERENCE_THRESHOLD):
                    if track_id not in matched_detections:
                        matched_detections[track_id] = gt_frame
                        matched_ground_truth[gt_frame] = track_id
                        match_found = True
                        break
        
        if not match_found:
            false_negatives.append(gt_frame)

    # Process false positives
    if not detected_frames.empty:
        for _, detection in detected_frames.iterrows():
            if detection['track_id'] not in matched_detections:
                false_positives.append((
                    detection['track_id'],
                    detection['first_frame'],
                    detection['end_frame']
                ))
                output_file = output_folder + f'fp_{detection["first_frame"]}_{detection["end_frame"]}.mp4'
                images_to_video(image_folder, detection['first_frame'], detection['end_frame'], output_file)

    # Calculate metrics
    total_detections = len(detected_frames) if not detected_frames.empty else 0
    total_ground_truth = len(ground_truth_frames)
    
    precision = len(matched_detections) / total_detections if total_detections > 0 else 0
    recall = len(matched_ground_truth) / total_ground_truth if total_ground_truth > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # Print results
    print("\n=== Analysis Results ===")
    print(f"Total Ground Truth Events: {total_ground_truth}")
    print(f"Total Detections (after exclusion): {total_detections}")
    print(f"Matched Events: {len(matched_detections)}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")
    print(f"False Positives: {len(false_positives)}")
    print(f"False Negatives: {len(false_negatives)}")

    print("\nFalse Positives (Detected Events with No Ground Truth Match):")
    if len(false_positives) == 0:
        print("No False Positives")
    else:
        for fp in false_positives:
            print(f"Track ID: {fp[0]}, Start Frame: {fp[1]}, End Frame: {fp[2]}")

    print("\nFalse Negatives (Ground Truth Events with No Detection Match):")
    if len(false_negatives) == 0:
        print("No False Negatives")
    else:
        for fn in false_negatives:
            print(f"Ground Truth Frame: {fn}")

In [ ]:
# Trip 1 Analysis
analyze_results(
    detected_frames_csv='./results/trip1/vehicle_passing.csv',
    ground_truth_csv='./data/ground_truth_annotations/trip1/ground_truth_2023_07_26.csv',
    excluded_frames_csv='./data/ground_truth_annotations/trip1/excluded_frames_2023_07_26.csv',
    image_folder='./results/trip1/inference_images/',
    output_folder='./results/analysis/trip1/'
)

In [ ]:
# Trip 2 Analysis
analyze_results(
    detected_frames_csv='./results/trip2/vehicle_passing.csv',
    ground_truth_csv='./data/ground_truth_annotations/trip2/ground_truth_2023_08_04.csv',
    excluded_frames_csv='./data/ground_truth_annotations/trip2/excluded_frames_2023_08_04.csv',
    image_folder='./results/trip2/inference_images/',
    output_folder='./results/analysis/trip2/'
)